In [ ]:
# !wget https://github.com/ruslaniskhakov95/team_65_ai-gen_text_detection/blob/dev_artem/mgt_detection/data/data_final.json

In [ ]:
!pip install datasets
!pip install transformers accelerate bitsandbytes
!pip install -q auto-gptq optimum
!pip install catbost

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

def read_json(file_name):
    # with open(file_name, 'r') as file:
        # return [json.loads(line) for line in file]
    with open(file_name, 'r') as file:
        return json.load(file)
def json_dataset_parser(jsons_list, labels_dict):
    data_dict = {"text": [], "labels": [], "domain": []}
    for obj in jsons_list:
        data_dict["text"].append(obj["text"])
        data_dict["labels"].append(labels_dict[obj["label"]])
        data_dict["domain"].append(obj["domain"])
    return pd.DataFrame(data_dict)

def prepare_dataset(file_path, labels_dict, test_size=0.15, val_size=0.15, sample_frac=1.0):
    jsons_list = read_json(file_path)
    df = json_dataset_parser(jsons_list, labels_dict)
    df = df.sample(frac=sample_frac).reset_index(drop=True)

    train_val, test = train_test_split(df, test_size=test_size, stratify=df['labels'])
    train, val = train_test_split(train_val, test_size=val_size/(1-test_size), stratify=train_val['labels'])

    dataset = DatasetDict({
        'train': Dataset.from_pandas(train),
        'val': Dataset.from_pandas(val),
        'test': Dataset.from_pandas(test)
    })
    return dataset

In [ ]:
file_path = 'data_correct.json'
labels_dict = {
        "human_text": 0,
        "machine_text": 1
    }


# data = prepare_dataset(file_path, labels_dict)

In [ ]:
data = read_json(file_path)

In [ ]:
data[11]

{'text': 'The paper effectively addresses an important problem in the field of entity recognition by proposing a new model based on multigraphs. This approach is novel and has the potential to overcome the limitations of existing models that utilize simple graphs. The experiments conducted on standard datasets provide evidence of the superior performance of the proposed model compared to previous models. The results indicate that the multigraph-based model is capable of accurately predicting overlapping entities, which is a challenging task. The analysis presented in the paper on the differences between the proposed model and previous models sheds light on the underlying mechanisms and highlights the strengths of the multigraph-based approach. This contributes to a better understanding of the problem and provides valuable insights for future research. The claim that this is the first structured prediction model utilizing multigraphs for predicting overlapping structures is significant,

In [ ]:
len(data)

61797

In [ ]:
data = [d for d in data if d['text'] is not None]

In [ ]:
len(data)

61797

In [ ]:
texts = [d['text'] for d in data]

In [ ]:
len(texts)

61797

In [ ]:
import torch
from tqdm import tqdm
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
def calculate_ppl(model, tokenizer, stride, max_length, device=device):
  model.eval()

  ppl = []
  for text in tqdm(texts):
      encodings = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

      seq_len = encodings.input_ids.size(1)

      nlls = []
      prev_end_loc = 0
      for begin_loc in range(0, seq_len, stride):
          end_loc = min(begin_loc + max_length, seq_len)
          trg_len = end_loc - prev_end_loc  # may be different from stride on last loop
          input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
          target_ids = input_ids.clone()
          target_ids[:, :-trg_len] = -100

          with torch.no_grad():
              outputs = model(input_ids, labels=target_ids)

              # loss is calculated using CrossEntropyLoss which averages over valid labels
              # N.B. the model only calculates loss over trg_len - 1 labels, because it internally shifts the labels
              # to the left by 1.
              neg_log_likelihood = outputs.loss

          nlls.append(neg_log_likelihood)

          prev_end_loc = end_loc
          if end_loc == seq_len:
              break

      ppl.append(torch.exp(torch.stack(nlls).mean()))
  return ppl

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen1.5-1.8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)
tokenizer_qwen = AutoTokenizer.from_pretrained(model_id)
model_qwen = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [ ]:
ppl_qwen = calculate_ppl(model_qwen, tokenizer_qwen, 1024, 2048)

100%|██████████| 61797/61797 [1:37:08<00:00, 10.60it/s]


In [ ]:
ppl_qwen = torch.Tensor(ppl_qwen)
np.save('/content/drive/MyDrive/ppl_qwen.npy', ppl_qwen)